In [7]:
import numpy as np 
import torch 
import pandas as pd 
import torch.nn.functional as F

In [ ]:
def kmeans(points: np.ndarray, k: int, eps: float = 1e-4):
    # 1. randomly initialize
    # centroids = points[np.random.choice(len(points), k, replace=False), :]

    # kmeans++
    centroids = np.zeros((k, points.shape[1]), dtype=float)
    centroids[0] = points[np.random.choice(len(points)), :]
    for i in range(1, k):
        dist = np.linalg.norm(points[:, np.newaxis] - centroids[:i], axis=2)   #   (N, i)
        min_sq_dist = dist.min(axis=1) ** 2
        prob = min_sq_dist / min_sq_dist.sum()
        c = np.random.choice(len(points), p=prob)
        centroids[i] = points[c, :]


    for _ in range(100):
        # 2. Assign points to clusters
        dist = np.linalg.norm(points[:, np.newaxis] - centroids, axis=2)   # (N, k)
        labels = np.argmin(dist, axis=1)   # (N, )

        # 3. Handle empty clusters
        for i in range(k):
            if np.sum(labels == i) == 0:
                c = np.random.choice(len(points))
                centroids[i, :] = points[c, :]
                labels[c] = i
        
        # 4. Recompute centroids
        new_centroids = np.array([points[labels == i, :].mean(axis=0) for i in range(k)])
        if np.max(np.linalg.norm(new_centroids - centroids, axis=1)) < eps:
            break
        centroids = new_centroids
    
    return centroids, labels

In [43]:
points = np.array([[1,0], [2,0], [3,0], [1,100], [2,100], [4, 100]], dtype=float)
kmeans(points, 2)

(array([[  2.        ,   0.        ],
        [  2.33333333, 100.        ]]),
 array([0, 0, 0, 1, 1, 1]))

In [13]:
kmeans(points, 6)

(array([[4., 4.],
        [4., 0.],
        [1., 4.],
        [4., 2.],
        [1., 0.],
        [1., 2.]], dtype=float32),
 array([5, 2, 4, 3, 0, 1]))

In [ ]:
def knn_predict_simple(X_train, y_train, X_test, k=5, task='classification'):
    """
    Simpler vectorized KNN (without scipy dependency).
    Uses manual mode calculation for classification.

    Parameters:
    -----------
    X_train : array-like, shape (n_train_samples, n_features)
        Training data
    y_train : array-like, shape (n_train_samples,)
        Training labels/values
    X_test : array-like, shape (n_test_samples, n_features)
        Test data
    """
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    
    # Compute distances using broadcasting
    # ||X_test[i] - X_train[j]||^2
    distances = np.sqrt(((X_test[:, np.newaxis, :] - X_train[np.newaxis, :, :]) ** 2).sum(axis=2))
    
    # Get k nearest neighbors
    k_nearest_indices = np.argsort(distances, axis=1)[:, :k]
    k_nearest_labels = y_train[k_nearest_indices]
    
    if task == 'classification':
        # Manual mode calculation using bincount
        predictions = np.array([
            np.bincount(k_nearest_labels[i].astype(int)).argmax() 
            for i in range(len(X_test))
        ])
    else:
        predictions = np.mean(k_nearest_labels, axis=1)
    
    return predictions

In [29]:
def kmeans_torch(points: torch.Tensor, k: int):
    # points: (n, d) array
    # 1. Randomly select k points as initial centroids
    centroids = points[torch.randperm(len(points))[:k], :]

    for _ in range(100):
        # 2. Assign each point to nearest centroid
        distances = torch.norm(points.unsqueeze(1) - centroids, dim=2)
        labels = torch.argmin(distances, dim=1)

        # 3. Recompute centroids
        # check if there is empty cluster
        for i in range(k):
            if torch.sum(labels == i).item() == 0:
                c = torch.randperm(len(points))[0]
                centroids[i] = points[c]
                labels[c] = i
        new_centroids = torch.stack([points[labels == i].mean(dim=0) for i in range(k)])
        if torch.min(torch.norm(new_centroids - centroids, dim=1)).item() < 1e-4:
            break
        centroids = new_centroids
    return centroids, labels

In [28]:
points = torch.tensor([[1, 2], [1, 4], [1, 0],
                   [4, 2], [4, 4], [4, 0]], dtype=torch.float32)
centroids = points[:2, :]
distances = torch.norm(points.unsqueeze(1) - centroids, dim=2)
labels = torch.argmin(distances, dim=1)
torch.stack([points[labels == i].mean(dim=0) for i in range(2)])

tensor([[2.5000, 1.0000],
        [2.5000, 4.0000]])

In [20]:
kmeans_torch(points, 2)

(tensor([[2.5000, 4.0000],
         [2.5000, 1.0000]]),
 tensor([1, 0, 1, 1, 0, 1]))

In [ ]:
def kmeans_plus(points: np.ndarray, k: int):
    # points: (n, d) array
    # 1. Improrved initialization of centroids using k-means++
    centroids = np.zeros((k, points.shape[1]), dtype=points.dtype)
    centroids[0] = points[np.random.choice(len(points))]
    for i in range(1, k):
        distances = np.linalg.norm(points[:, np.newaxis] - centroids[:i], axis=2)
        min_squared_distances = np.min(distances, axis=1) ** 2
        probs = min_squared_distances / np.sum(min_squared_distances)
        centroids[i] = points[np.random.choice(len(points), p=probs)]

    for _ in range(100):
        # 2. Assign each point to nearest centroid
        distances = np.linalg.norm(points[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)

        # 3. Recompute centroids
        # check if there is empty cluster
        for i in range(k):
            if np.sum(labels == i) == 0:
                c = np.random.choice(len(points))
                centroids[i] = points[c]
                labels[c] = i
        new_centroids = np.array([points[labels == i].mean(axis=0) for i in range(k)])
        if np.max(np.linalg.norm(new_centroids - centroids, axis=1)) < 1e-4:
            break
        centroids = new_centroids
    
    return centroids, labels

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

def spherical_kmeans(X, n_clusters, max_iter=100):
    # X should already be normalized (on unit sphere)
    X = normalize(X, norm='l2')
    
    # Initialize centroids randomly
    indices = np.random.choice(len(X), n_clusters, replace=False)
    centroids = X[indices].copy()
    
    for _ in range(max_iter):
        # Assignment: use cosine similarity (dot product for unit vectors)
        similarities = X @ centroids.T
        labels = np.argmax(similarities, axis=1)
        
        # Update: compute mean and normalize back to sphere
        new_centroids = np.array([
            X[labels == k].mean(axis=0) 
            for k in range(n_clusters)
        ])
        centroids = normalize(new_centroids, norm='l2')
        
    return labels, centroids

In [8]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.head_size = d_model // num_heads
        self.num_heads = num_heads

        self.Wq = torch.nn.Linear(d_model, d_model)
        self.Wk = torch.nn.Linear(d_model, d_model)
        self.Wv = torch.nn.Linear(d_model, d_model)
        self.Wo = torch.nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor):
        # x.shape = (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape
        Q = self.Wq(x).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2) # (batch_size, num_heads, seq_len, head_size)
        K = self.Wk(x).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2) 
        V = self.Wv(x).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_size) # (batch_size, num_heads, seq_len, seq_len)
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)  # shape: (batch_size, num_heads, seq_len, seq_len)
        attn_output = torch.matmul(attn_weights, V)  # shape: (batch_size, num_heads, seq_len, head_size)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)  # shape: (batch_size, seq_len, d_model)
        output = self.Wo(attn_output)  # shape: (batch_size, seq_len, d_model)
        return output


In [9]:
class FFN(torch.nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.linear1 = torch.nn.Linear(d_model, d_ff)
        self.linear2 = torch.nn.Linear(d_ff, d_model)
        self.relu = torch.nn.ReLU()

    def forward(self, x: torch.Tensor):
        return self.linear2(self.relu(self.linear1(x)))

In [10]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor):
        # x.shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

In [11]:
class Transformer(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FFN(d_model, d_ff)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.pe = PositionalEncoding(d_model)

    def forward(self, x: torch.Tensor):
        # Multi-head attention sublayer
        # Pre-norm
        x = self.pe(x)
        x = self.norm1(x)
        attn_output = self.mha(x)
        x = x + attn_output  # Add & Norm
        # attn_output = self.mha(x)
        # x = self.norm1(x + attn_output)  # Add & Norm

        # Feed-forward sublayer
        x = self.norm2(x)
        ffn_output = self.ffn(x)
        x = x + ffn_output 
        # x = self.norm2(x + ffn_output)  # Add & Norm

        return x

In [12]:
class KVCache:
    def __init__(self):
        self.key_cache = None
        self.value_cache = None
    
    def update(self, keys: torch.Tensor, values: torch.Tensor):
        """
        key.shape = [batch_size, num_heads, seq_len, head_size]
        value.shape = [batch_size, num_heads, seq_len, head_size]
        concat on seq_len dimension
        """
        if self.key_cache is None:
            self.key_cache = keys
            self.value_cache = values
        else:
            self.key_cache = torch.cat([self.key_cache, keys], dim=2)
            self.value_cache = torch.cat([self.value_cache, values], dim=2)
        return self.key_cache, self.value_cache

    def get(self):
        return self.key_cache, self.value_cache
    
    def clear(self):
        self.key_cache = None
        self.value_cache = None

In [13]:
class MultiHeadAttentionWithCache(torch.nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0 
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_size = d_model // n_heads

        self.q_proj = torch.nn.Linear(d_model, d_model)
        self.k_proj = torch.nn.Linear(d_model, d_model)
        self.v_proj = torch.nn.Linear(d_model, d_model)
        self.o_proj = torch.nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, kv_cache: KVCache | None = None):
        """
        seq_len = length of Q
        total_seq_len = length of K, V (including cached)
        """
        batch_size, seq_len, _ = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_size).transpose(1, 2)  # (batch_size, n_heads, seq_len, head_size)
        K = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_size).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_size).transpose(1, 2)

        if kv_cache is not None:
            K, V = kv_cache.update(K, V)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_size)  # (batch_size, n_heads, seq_len, total_seq_len)
        attn_weights = F.softmax(scores, dim=-1)  # shape: (batch_size, n_heads, seq_len, total_seq_len)
        attn_output = torch.matmul(attn_weights, V)  # shape: (batch_size, n_heads, seq_len, head_size)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)  # shape: (batch_size, seq_len, d_model)
        output = self.o_proj(attn_output)  # shape: (batch_size, seq_len, d_model)
        return output


In [21]:
class RotaryPositionalEncoding:
    def __init__(self, dim: int, max_len: int = 5000):
        self.dim = dim
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-np.log(10000.0) / dim))
        self.sinusoid_table = torch.zeros(max_len, dim // 2, 2)
        self.sinusoid_table[:, :, 0] = torch.sin(position * div_term)
        self.sinusoid_table[:, :, 1] = torch.cos(position * div_term)

    def get_embedding(self, seq_len: int):
        return self.sinusoid_table[:seq_len, :, :]

    def apply_rotary_pos_emb(self, x: torch.Tensor):
        """
        x: (batch_size, num_heads, seq_len, head_size)
        """
        seq_len = x.size(2)
        sinusoid = self.get_embedding(seq_len).to(x.device)  # (seq_len, dim // 2, 2)
        sinusoid = sinusoid.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, dim // 2, 2)
        x = x.reshape(*x.shape[:-1], self.dim // 2, 2)  # (batch_size, num_heads, seq_len, dim // 2, 2)
        x1 = x[..., 0]
        x2 = x[..., 1]
        sin = sinusoid[..., 0]
        cos = sinusoid[..., 1]
        x_rotated = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
        # print(x_rotated.shape)
        # x_rotated = x_rotated.reshape(*x.shape[:-2], self.dim)  # (batch_size, num_heads, seq_len, dim)
        return x_rotated


In [22]:
rope = RotaryPositionalEncoding(4)
x = torch.ones(2, 3, 5, 4)
rope.apply_rotary_pos_emb(x).shape

torch.Size([2, 3, 5, 4])

In [44]:
def BPE(tokens: list[int], k: int) -> dict:
    """
    Input: A list of token IDs and vocab size 
    Output: {(merged_token1, merged_token2): new_token_id}
    """
    from collections import Counter

    new_token_id = max(tokens)
    merges = {}

    for _ in range(k):
        pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        pair_counts = Counter(pairs)
        if not pair_counts:
            break
        most_common_pair, _ = pair_counts.most_common(1)[0]
        new_token_id += 1
        merges[most_common_pair] = new_token_id

        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == most_common_pair:
                new_tokens.append(new_token_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return merges

In [ ]:
def tokenize(tokens: list[int], merges: dict) -> list[int]:
    """
    Input: A list of token IDs and merges dict
    Output: A list of token IDs after applying merges
    """
    
    while len(tokens) >= 2:
        min_merge_id = None
        min_index = None
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            if pair in merges:
                merge_id = merges[pair]
                if min_merge_id is None or merge_id < min_merge_id:
                    min_merge_id = merge_id
                    min_index = i
        if min_merge_id is None:
            break
        pair = (tokens[min_index], tokens[min_index + 1])
        new_tokens = []
        i = 0
        while i < len(tokens):
            if tokens[i] == pair[0] and i < len(tokens) - 1 and tokens[i + 1] == pair[1]:
                new_tokens.append(min_merge_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

In [ ]:
def contrastive_loss(embed_1: torch.Tensor, embed_2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """
    embed_1: (batch_size, dim)
    embed_2: (batch_size, dim)
    """
    batch_size = embed_1.size(0)
    embed_1 = F.normalize(embed_1, dim=1)
    embed_2 = F.normalize(embed_2, dim=1)

    logits = torch.matmul(embed_1, embed_2.T) / temperature  # (batch_size, batch_size)
    labels = torch.arange(batch_size).to(embed_1.device)

    loss_1 = F.cross_entropy(logits, labels)
    loss_2 = F.cross_entropy(logits.T, labels)
    loss = (loss_1 + loss_2) / 2
    return loss

In [ ]:
def mergeKLists(self, lists):
    x = []
    class Index:
        def __init__(self, i):
            self.i = i

        def __lt__(self, other):
            return x[self.i].val < x[other.i].val

    import heapq
    n = len(lists)
    for lst in lists:
        if lst is None:
            continue
        x.append(lst)
    
    head = None
    tail = None
    heap = []
    for i in range(len(x)):
        heap.append(Index(i))
    heapq.heapify(heap)

    while len(heap) > 0:
        idx = heapq.heappop(heap)
        idx = idx.i
        if head is None:
            head = x[idx]
            tail = x[idx]
        else:
            tail.next = x[idx]
            tail = x[idx]
        a = x[idx]
        x[idx] = x[idx].next
        if x[idx] is not None:
            heapq.heappush(heap, Index(idx))
        a.next = None

    return head

In [ ]:
from sortedcontainers import SortedList

class Event:
    def __init__(self, time, event, amount):
        """ 
        Event:
        1 - Add credit
        2 - Use credit
        3 - Expire credit
        """
        self.time = time
        self.event = event
        self.amount = amount

    def __lt__(self, other):
        return self.time < other.time


class GPUCredit:
    def __init__(self):
        self.events = SortedList()

    def addCredit(self, creditID: str, amount: int, timestamp: int, expiration: int):
        self.events.add(Event(timestamp, 1, amount))
        self.events.add(Event(timestamp + expiration + 1, 3, amount))

    def getBalance(self, timestamp: int) -> int:
        # print("timestamp =", timestamp)
        balance = 0
        used_credit = 0
        for event in self.events:
            if event.time > timestamp:
                break
            if event.event == 1:
                balance += event.amount
            elif event.event == 2:
                used_credit += event.amount
            elif event.event == 3:
                balance -= event.amount
                used_credit = max(0, used_credit - event.amount)
            # print(event)
            # print("balance =", balance, "used_credit =", used_credit)
        return balance - used_credit
    
    def useCredit(self, amount: int, timestamp: int):
        if self.getBalance(timestamp) < amount:
            return False
        self.events.add(Event(timestamp, 2, amount))
        return True

In [2]:
gpuCredit = GPUCredit()
gpuCredit.addCredit('amazon', 40, 10, 50)
gpuCredit.useCredit(30, 30)
print(gpuCredit.getBalance(40)) # returns 10
gpuCredit.addCredit('google', 20, 60, 10)
print(gpuCredit.getBalance(60)) # returns 30
print(gpuCredit.getBalance(61)) # returns 20
print(gpuCredit.getBalance(70)) # returns 20
print(gpuCredit.getBalance(71)) # returns None

10
30
20
20
0


In [3]:
import torch 
inputs = torch.tensor([[1.0, 2.0], [2.0, 3.0], [3.0, 4.0]], dtype=torch.float32)
targets = torch.tensor([1.0, 2.5, 3.5], dtype=torch.float32)
epochs = 100
lr = 0.01

In [5]:
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(2, 1)
        torch.nn.init.normal_(self.linear.weight)
        torch.nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.linear(x).squeeze()
    
model = Net()
for epoch in range(epochs):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = torch.nn.functional.mse_loss(outputs, targets)
    loss.backward()
    optimizer.step()

In [6]:
model.linear.weight, model.linear.bias

(Parameter containing:
 tensor([[ 1.4956, -0.1821]], requires_grad=True),
 Parameter containing:
 tensor([-0.1269], requires_grad=True))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ConvNet, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 48x48x32
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 24x24x64
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # 12x12x128
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)  # 6x6x256
        self.bn4 = nn.BatchNorm2d(256)
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.fc2 = nn.Linear(512, num_classes)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 48 -> 24
        
        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 24 -> 12
        
        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 12 -> 6
        
        # Conv block 4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # 6 -> 3
        
        # Flatten
        x = x.view(x.size(0), -1)  # Flatten to [batch_size, 256*3*3]
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [ ]:
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader
from torch.distributed.algorithms.join import Join

# Initialize process group (all processes run this)
dist.init_process_group(backend='nccl')
rank = dist.get_rank()  # Unique ID for this process
world_size = dist.get_world_size()  # Total number of processes

# Same model on each GPU
model = MyModel().to(rank)
model = DDP(model, device_ids=[rank])

# Each process gets different data
sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
dataloader = DataLoader(dataset, sampler=sampler)

# Training loop - same code, different data
with Join([model, optim]):
    for batch in dataloader:
        loss = model(batch)
        loss.backward()  # Gradients automatically synchronized across processes
        optimizer.step()